# Multilingual retrieval and evaluation

Compare keyword search, E5 and hybrid ranking using the shared live pipeline.

<a id="setup"></a>
## 1. Shared project environment

This notebook isolates retrieval and evaluation. It calls the same code as the app and does not train E5.

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "news_pipeline.py").exists())
sys.path.insert(0, str(ROOT))
os.environ.setdefault("NEWS_EMBEDDING_DEVICE", "cpu")
from src.portfolio_config import ARTIFACTS, BASE_MODEL, EMBEDDING_MODEL, configure_cache, read_json, read_jsonl
configure_cache()
from src.portfolio_reports import evidence_report
print("Project:", ROOT)
print("Artifacts:", ARTIFACTS)


<a id="retrieval-code"></a>
## 2. Inspect retrieval and explanations

Article vectors are saved once per model/content version. Each new interest gets a new vector. Exact filters and semantic similarities remain separate visible contributions. Phrase matches are similarity probes, not hidden reasoning.

In [ ]:
import inspect
from src.news_pipeline import NewsPipeline
print(inspect.getsource(NewsPipeline.recommend))
print(inspect.getsource(NewsPipeline.explain))

<a id="retrieval-run"></a>
## 3. Compare three ranking methods

TF-IDF is the keyword baseline, E5 is multilingual semantic retrieval, and hybrid adds exact required-category coverage. Without required filters E5 and hybrid should agree. Scores are not percentages of confidence.

In [ ]:
pipeline = NewsPipeline()
interest = "Iran nuclear diplomacy and sanctions"
required_filters = {"countries": ["Iran"]}
for method in ("keyword", "e5", "hybrid"):
    display({"method": method, "ranking": pipeline.recommend(interest, required_filters, method=method, limit=5)})

<a id="retrieval-change"></a>
## 4. Change the input, not the stored result

Try another interest. The ranking is recalculated using the same processed corpus. If no version-matching index exists, the explicit empty status tells you to rebuild it.

In [ ]:
display(pipeline.recommend("Lebanon and regional diplomacy", limit=5))

<a id="retrieval-review"></a>
## 5. Frozen relevance judgments

Review thirty query/article pairs, graded 0, 1 or 2. Evaluation reranks each ten-article judged pool; it is not a claim of full-corpus Recall. Unjudged articles must never become automatic negatives. Selection and query roles are frozen before grading.

In [ ]:
path = ARTIFACTS / "review" / "ranking.jsonl"
judgments = read_jsonl(path) if path.exists() else []
display(judgments[:2])
print("Judgments:", len(judgments))
display(read_json(ARTIFACTS / "recommendation_evaluation.json", {"status": "pending"}))

<a id="retrieval-metrics"></a>
## 6. Read how the metrics are calculated

Recall@5 asks how many judged relevant articles reached the top five. nDCG@5 rewards putting highly relevant articles nearer the top. No reviewed judgments means no claimed recommendation quality score.

In [ ]:
from src.news_evaluation import ranking_metrics, ndcg_at_k
print(inspect.getsource(ranking_metrics))
print(inspect.getsource(ndcg_at_k))
display(evidence_report())